# Two sentences and their pairing

Write two tag-augmented sentences, draw their graphs, and show the optimal FGW pairing between their cues.

Tags: `{f: ...}` facial expression, `{g: ...}` gesture (name from `data/gestures.csv`), `[...]` prosody, `{t: ...}` linguistic marker (written after the word it marks).

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
from sentence_transformers import SentenceTransformer

sys.path.append(os.path.realpath("../lib"))
from graph_multimodalsentence import Graph, draw_coupling, draw_sentence_graph
from ot_distances import Fused_Gromov_Wasserstein_distance

embedding = SentenceTransformer("sentence-transformers/bert-base-nli-mean-tokens")

## Sentences and parameters

In [ ]:
sentence1 = ("[thoughtful] {f: confidence} {g: Acknowledging} Yes {t: Affirmation} take the left route. "
             "{g: Arm Gesture (Left)} {f: neutral} Stay alert and {t: Conjunction.Start} move steadily.")
sentence2 = ("{f: confidence} {g: Arm Gesture (Left)} [thoughtful] Yes {t: Affirmation} , take the left route. "
             "{g: Head Nod Yes} {f: neutral} Stay alert and {t: Conjunction.Start} move steadily.")

ALPHA = 0.14      # 0: match cue meanings only, 1: match timing only
THRESH = 0.1      # hide links carrying less than this fraction of the largest mass

## Graphs

In [ ]:
g1 = Graph.from_sentence(sentence1, embedding)
g2 = Graph.from_sentence(sentence2, embedding)

draw_sentence_graph(g1, title="Sentence 1")
plt.show()
draw_sentence_graph(g2, title="Sentence 2")
plt.show()

## Pairing

In [ ]:
fgw = Fused_Gromov_Wasserstein_distance(alpha=ALPHA, features_metric="sqeuclidean", method="temporal")
distance = fgw.graph_d(g1, g2)

draw_coupling(g1, g2, fgw.transp, thresh=THRESH, distance=distance)
plt.savefig("pairing.png", dpi=300, bbox_inches="tight")
plt.show()

## Matched cues

For each node of sentence 1 (cues and start node), the node of sentence 2 receiving most of its mass, and the share of its mass sent there.

In [ ]:
transp = fgw.transp
pd.DataFrame([{
    "sentence 1": cue.label,
    "sentence 2": g2.node_cues()[transp[i].argmax()].label,
    "share of mass": transp[i].max() / transp[i].sum(),
} for i, cue in enumerate(g1.node_cues())]).round(2)